## Notebook 02 : branch update for Maxwell branch k
    1. The Maxwell evolution equation.
    2. Exact solution over one step (integrating factor, τ frozen at τ_bar ) 
    3. Midpoint rule for the integral -> weights a and b

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import vcann_core as vc

# Plot style
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve().parent  # notebook lives in <repo>/non-isothermal-vCANN
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from fenicsx.models.plotstyle import C, SERIF, apply_style, style_axes, style_legend
apply_style()



## Relaxation time of the step: $\tau_k$ frozen at the endpoint mean $\bar\tau_k$

We are replacing $\tau_k(t)$, which in reality varies, with a constant. This constant is the endpoint mean,
$$
\bar\tau_k = \tfrac12\big(\tau_{k,n} + \tau_{k,n+1}\big),
$$
roughly the mid-step value. This is not the same as the midpoint rule for $b$ in the next cell, where the kernel is evaluated at $s=t_n+\Delta t/2$


In [ ]:
def update_tau(params, T_k, tau_n, C_bar_new, T_hat_new):
    tau_new_k = vc.tau_k(params, T_k, C_bar_new, T_hat_new)
    return 0.5*(tau_n + tau_new_k), tau_new_k

Note regarding the kernel: "kernel" is the standard name for the weight funciton inside an integral of the form
$$
\int K(t, s)\,f(s)\,\mathrm ds
$$
It says how much each contribution $f(s)$ counts in the result at time t. In our integral:
$$
\int_{t_n}^{t_{n+1}} \underbrace{e^{-(t_{n+1}-s)/\bar\tau_k}}_{\text{kernel } K}\;\underbrace{\dot{\bar{\mathbf S}}_k(s)}_{f(s)}\,\mathrm ds
$$

- $f(s) = \dot{\bar {S}}_k (s)$: the loading the branch receives at time s.
- **The kernel**: how much of that loading is still remembered at $t_{n+1}$. It is the relaxation function: a contribution made at $s$ has been relaxing for $t_{n+1} - s$

## Decaying weights: midpoint rule

$$
a = e^{-\Delta t/\bar\tau_k}, \qquad b = e^{-\Delta t/(2\bar\tau_k)}
$$

In [ ]:
def decaying_weights(tau_bar, dt):
    return jnp.exp(-dt/tau_bar), jnp.exp(-dt/(2*tau_bar))


## Update the overstress: $\mathbf {Q}_{k, n+1}$

$$
\mathbf Q_{k,n+1} = a\,\mathbf Q_{k,n} + b\,\big(\bar{\mathbf S}_{k,n+1} - \bar{\mathbf S}_{k,n}\big)
$$

In [ ]:
def update_Q(Q_n, S_bar_n, S_bar_new, a, b):
    return a*Q_n + b*(S_bar_new - S_bar_n)

## The tangent: $\mathbb {\bar C}_k$

$$
\bar{\mathbb C}_k = 4\,\frac{\partial^2\Psi_k}{\partial\bar{\mathbf C}\,\partial\bar{\mathbf C}}\Big|_{(\bar{\mathbf C}_{n+1},\,\hat T_{n+1})}
$$

In [ ]:
def branch_tangent(psi_parameters, C_bar_new, T_hat_new):
    return 4*jax.hessian(vc.psi_corrected, argnums=1)(psi_parameters, C_bar_new, T_hat_new)

## Branch stress: $\mathbf {\bar S}_k^{neq}$

$$
\bar{\mathbf S}_k^{\mathrm{neq}} = \frac{1}{2\mu_k}\,\bar{\mathbb C}_k\big(\bar{\mathbf C}_{n+1}, \hat T_{n+1}\big) : \mathbf Q_{k,n+1}
$$

In [ ]:
MU_K = 1.0

def branch_stress(CC_k, Q_new, MU_K):
    return 1/(2*MU_K)*jnp.tensordot(CC_k, Q_new, axes=2) 

## Branch Update

In [ ]:
def branch_update(psi_p, tau_p, T_k, MU_K, H_k_n, C_bar_new, T_hat_new, dt):
    
    S_bar_k_new = vc.S_bar_k(psi_p, C_bar_new, T_hat_new)
    tau_k_bar, tau_k_new = update_tau(tau_p, T_k, H_k_n["tau"], C_bar_new, T_hat_new)
    a_k, b_k = decaying_weights(tau_k_bar, dt)
    Q_k_new = update_Q(H_k_n["Q"], H_k_n["S_bar"], S_bar_k_new, a_k, b_k)
    CC_k = branch_tangent(psi_p, C_bar_new, T_hat_new)
    S_bar_k_neq = branch_stress(CC_k, Q_k_new, MU_K)

    return dict(Q=Q_k_new, S_bar=S_bar_k_new, tau=tau_k_new), S_bar_k_neq

## Initial history

In [ ]:
def initial_history(psi_p, tau_p, T_k, C_bar_initial, T_hat_initial):
    
    Q_initial = jnp.zeros((3, 3))
    S_bar_initial = vc.S_bar_k(psi_p, C_bar_initial, T_hat_initial)
    tau_initial = vc.tau_k(tau_p, T_k, C_bar_initial, T_hat_initial)

    return dict(Q=Q_initial, S_bar=S_bar_initial, tau=tau_initial)


## Test 1: Relaxation

In [ ]:
print(jax.devices())
print(jax.default_backend()) 

In [ ]:
LAMBDA = 1.2

F_initial = jnp.eye(3)
F_final = jnp.diag(jnp.array([LAMBDA, 1/jnp.sqrt(LAMBDA), 1/jnp.sqrt(LAMBDA)]))
print(f"Final deformation:")
print(F_final)
print("--------------------------------------------------------------------------")

C_initial = vc.right_cauchy_green(F_initial)
print(f"Initial right Cauchy-Green tensor:")
print(C_initial)
print("--------------------------------------------------------------------------")

C_bar_initial, J = vc.isochoric_C(C_initial)
print(f"Initial isochoric right Cauchy-Green tensor:")
print(C_bar_initial, J)
print("--------------------------------------------------------------------------")


C_final = vc.right_cauchy_green(F_final)
print(f"Final right Cauchy-Green tensor:")
print(C_final)
print("--------------------------------------------------------------------------")

C_bar_final, J = vc.isochoric_C(C_final)
print(f"Final isochoric right Cauchy-Green tensor:")
print(C_bar_final, J)
print("--------------------------------------------------------------------------")

master_key = jax.random.PRNGKey(0)
print(f"Master key:")
print(master_key)
print("--------------------------------------------------------------------------")

key_psi, key_tau = jax.random.split(master_key)
key_psi_split = jax.random.split(key_psi, vc.N_MB)
key_tau_split = jax.random.split(key_tau, vc.N_MB)

print(f"Splitted keys for Psi:")
print(key_psi_split)
print("--------------------------------------------------------------------------")

print(f"Splitted keys for tau:")
print(key_tau_split)
print("--------------------------------------------------------------------------")


In [ ]:
T_hat_initial = 0.0

psi_params = [vc.init_psi_params(k) for k in key_psi_split]
tau_params = [vc.init_tau_params(k) for k in key_tau_split]

print(psi_params)

In [ ]:
H_0 = [initial_history(psi_params[k], tau_params[k], vc.T_K[k], C_bar_initial, T_hat_initial) for k in range(vc.N_MB)]
print(H_0)
print(H_0[2]["tau"])

## Test 1: ramp + hold (just branch 3 with $\tau_k$ ≈ 0.6 s)

In [ ]:
# Lets start with just one branch:
k = 2                           # branch 3: τ ≈ 0.6 s, easy to watch. 
T_hat = 0.0                     # Temperature fixed for the whole test.
n_ramp, dt_ramp = 10, 0.1       # ramp: 10 steps of 0.1 s each.
n_hold, dt_hold = 100, 0.05     # hold: 100 steps of 0.05 s (5 s ≈ 8τ) 

In [ ]:
# The ramp:

H = H_0[k]
Qs_ramp = [H["Q"]]
for n in range(1, n_ramp + 1):
    lam = 1.0 + 0.2 *n / n_ramp 
    F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
    C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
    H, S_bar_k_neq = branch_update(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_ramp)
    Qs_ramp.append(H["Q"])
    print(S_bar_k_neq)

Qs_ramp = jnp.stack(Qs_ramp)

print("-----------------------------------------------------------")
print(H["Q"])
print("Qs ramp shape:", Qs_ramp.shape)

In [ ]:
# Now we save the starting point of the hold.

Q_hold = H["Q"]             # overstress at the start of the hold.     
tau_hold = H["tau"]         # tau at the stretched state (constant during the hold)

print(f"τ_k initial:", H_0[k]["tau"])
print(f"τ_k after stretched:", tau_hold)
print(f"Δτ:", tau_hold - H_0[k]["tau"])


In [ ]:
# The hold: Same C_bar, same T_hat: just keep steeping and record Q

Qs_hold = []
for m in range(1, n_hold + 1):
    H, S_bar_k_neq = branch_update(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_hold)
    Qs_hold.append(H["Q"])
Qs_hold = jnp.stack(Qs_hold)          # shape (100, 3, 3)
print(Qs_hold[:10, :, :])
print(Qs_hold[-1, :, :])

**`jnp.arange(start, stop)`** returns the integers from `start` up to, but **not including**, `stop`:
`jnp.arange(1, 5)` → `[1, 2, 3, 4]`.

Here, `jnp.arange(1, n_hold + 1)` gives the hold-step numbers $m = 1, \ldots, n_{\text{hold}}$, and multiplying by the constant step gives the time since the start of the hold:

$$
t_m - t_{\mathrm h} = m\,\Delta t_{\text{hold}}, \qquad m = 1, \ldots, n_{\text{hold}}
$$

In [ ]:
# Now it is time to compare with the exact solution:

t_hold = dt_hold * jnp.arange(1, n_hold + 1)        # 0.05, 0.10, ..., 5.0
print(t_hold.shape)

Q_exact = jnp.exp(-t_hold/ tau_hold)[:, None, None]*Q_hold
err = jnp.abs(Qs_hold - Q_exact).max()/jnp.abs(Q_hold).max()
err_vec = jnp.abs(Qs_hold - Q_exact)/jnp.abs(Q_hold).max()

print(err_vec.shape)
print("max relative error:", err)
print("relative errors:", err_vec[:10,:,:])


On the ramp above: it is interesting to see how it starts to relax, bending instead of being linear during the whole ramp. It should reach around 7 MPa, but it only gets slightly above 5 MPa due to that relaxation ($\tau_k=0.596 s$)

## Test 1: ramp + hold (every branch)

In [ ]:
T_hat = 0.0
LAMBDA = 1.2
n_ramp, dt_ramp = 10, 0.1                                                       # ramp: 1 s
n_hold, dt_hold = 100, 0.05                                                     # hold: 5 s, constant step
lams   = 1.0 + (LAMBDA - 1.0) * jnp.arange(1, n_ramp + 1) / n_ramp              # 1.02, ..., 1.20
t_ramp = dt_ramp * jnp.arange(0, n_ramp + 1)                                    # 0.0, ..., 1.0  (11 values, incl. t = 0)
t_hold = dt_hold * jnp.arange(1, n_hold + 1)                                    # time since start of hold (100 values)

In [ ]:
# Plotting Q11 exact, numerical and T over t:

import matplotlib.pyplot as plt

t_h    = n_ramp * dt_ramp
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)
Q_fine = jnp.exp(-t_fine / tau_hold) * Q_hold[0, 0]

fig, (ax, axl) = plt.subplots(1, 2, figsize=(12, 4))

# ---- left: whole history, linear axis ----
ax.plot(t_ramp, Qs_ramp[:, 0, 0], "o", color="tab:red", label="Ramp (numerical)")
ax.plot(t_h + t_hold, Qs_hold[:, 0, 0], "s", mfc="none", color="tab:blue", label="Hold (numerical)")
ax.plot(t_h + t_fine, Q_fine, "k-", lw=1, label=r"Exact: $Q_{k,\mathrm{h}}\,e^{-(t-t_\mathrm{h})/\tau_k}$")
ax.axvline(t_h, color="k", ls="--", lw=1)
ax.set_xlabel(r"Time $t$ [s]")
ax.set_ylabel(r"$Q_{k,11}$ [MPa]")
ax.grid(True, alpha=0.3)

ax2 = ax.twinx()
ax2.plot([0.0, t_h + n_hold * dt_hold], [T_hat, T_hat], color="tab:orange", label=r"$\hat T$")
ax2.set_ylim(-1.0, 1.0)
ax2.set_ylabel(r"$\hat T$ [-]")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2)

# ---- right: hold only, logarithmic Q axis ----
axl.semilogy(t_hold, Qs_hold[:, 0, 0], "s", mfc="none", color="tab:blue", label="Hold (numerical)")
axl.semilogy(t_fine, Q_fine, "k-", lw=1, label="Exact")
axl.set_xlabel(r"Hold time $t - t_\mathrm{h}$ [s]")
axl.set_ylabel(r"$Q_{k,11}$ [MPa] (log scale)")
axl.set_title(rf"Straight line, slope $-1/\tau_k$, $\tau_k = {float(tau_hold):.3f}$ s")
axl.grid(True, which="both", alpha=0.3)
axl.legend()

fig.tight_layout()
fig.savefig("test1_relaxation.pdf"); fig.savefig("test1_relaxation.png", dpi=500)

In [ ]:
# Compile branch_update using jax.jit. Every call inside the for loops will reuse this compiled version, being much faster.
# In case I modify branch_update I need to recompile because if not, bu will still be the older version.

bu = jax.jit(branch_update)

In [ ]:
results = []
for k in range(vc.N_MB):
    H = H_0[k]
    Q_ramp, S_ramp = [H["Q"]], [H["S_bar"]]                                                                    # include t = 0
    for lam in lams:
        F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
        C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
        H, _ = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_ramp)
        Q_ramp.append(H["Q"]); S_ramp.append(H["S_bar"])
    Q_h, tau_h = H["Q"], H["tau"]                                                                              # start of the hold
    Q_hold = []
    for m in range(n_hold):
        H, _ = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_hold)
        Q_hold.append(H["Q"])
    results.append(dict(Q_ramp=jnp.stack(Q_ramp), S_ramp=jnp.stack(S_ramp),
                        Q_hold=jnp.stack(Q_hold), Q_h=Q_h, tau_h=tau_h))


In [ ]:
for k, r in enumerate(results):
    Q_exact = jnp.exp(-t_hold / r["tau_h"])[:, None, None] * r["Q_h"]
    err = jnp.abs(r["Q_hold"] - Q_exact).max() / jnp.abs(r["Q_h"]).max()
    print(f"Branch {k+1}: tau_h = {float(r['tau_h']):.4g} s,  max rel. error = {float(err):.2e}")

In [ ]:
t_h    = n_ramp * dt_ramp
t_end  = t_h + n_hold * dt_hold
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)

fig, axes = plt.subplots(1, vc.N_MB, figsize=(4.2 * vc.N_MB, 3.8))
for k, (ax, r) in enumerate(zip(axes, results)):
    Q_fine = jnp.exp(-t_fine / r["tau_h"]) * r["Q_h"][0, 0]
    ax.plot(t_ramp, r["Q_ramp"][:, 0, 0], "o", color="tab:red", ms=4, label="Ramp (numerical)")
    ax.plot(t_h + t_hold[::5], r["Q_hold"][::5, 0, 0], "s", mfc="none", color="tab:blue", ms=4,
            label="Hold (numerical)")
    ax.plot(t_h + t_fine, Q_fine, "k-", lw=1, label="Exact")
    ax.axvline(t_h, color="k", ls="--", lw=1)
    ax.set_title(rf"Branch {k+1}, $\tau_k = {float(r['tau_h']):.3g}$ s")
    ax.set_xlabel(r"Time $t$ [s]")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot([0.0, t_end], [T_hat, T_hat], color="tab:orange", label=r"$\hat T$")
    ax2.set_ylim(-1.0, 1.0)
    if k < vc.N_MB - 1:
        ax2.set_yticklabels([])                     # T axis labelled only on the last panel
    else:
        ax2.set_ylabel(r"$\hat T$ [-]")

axes[0].set_ylabel(r"$Q_{k,11}$ [MPa]")
h1, l1 = axes[0].get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
axes[0].legend(h1 + h2, l1 + l2, fontsize=8)
fig.tight_layout()
fig.savefig("test1_all_branches.pdf"); fig.savefig("test1_all_branches.png", dpi=200)

In [ ]:
t_h    = n_ramp * dt_ramp
t_end  = t_h + n_hold * dt_hold
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)

# loading history lambda(t): 1 at t = 0, the ramp values, then constant during the hold
t_lam = jnp.concatenate([t_ramp, jnp.array([t_end])])
lam_t = jnp.concatenate([jnp.array([1.0]), lams, jnp.array([LAMBDA])])

fig, axes = plt.subplots(1, vc.N_MB, figsize=(4.4 * vc.N_MB, 3.8))
for k, (ax, r) in enumerate(zip(axes, results)):
    Q_fine = jnp.exp(-t_fine / r["tau_h"]) * r["Q_h"][0, 0]
    ax.plot(t_ramp, r["Q_ramp"][:, 0, 0], "o-", color="tab:red", ms=4, label="Ramp (numerical)")
    ax.plot(t_h + t_hold[::5], r["Q_hold"][::5, 0, 0], "s", mfc="none", color="tab:blue", ms=4,
            label="Hold (numerical)")
    ax.plot(t_h + t_fine, Q_fine, "tab:blue", lw=1, label="Exact")
    # ax.axvline(t_h, color="k", ls="--", lw=1)
    ax.set_title(rf"Branch {k+1}, $\tau_k = {float(r['tau_h']):.3g}$ s")
    ax.set_xlabel(r"Time $t$ [s]")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()                                    # 2nd y-axis: temperature
    ax2.plot([0.0, t_end], [T_hat, T_hat], color="tab:orange", label=r"$\hat T$")
    ax2.set_ylim(-1.0, 1.0)

    ax3 = ax.twinx()                                    # 3rd y-axis: stretch
    ax3.spines["right"].set_position(("axes", 1.22))    # move its axis line further right
    ax3.plot(t_lam, lam_t, color="tab:gray", ls="--", lw=1.0, label=r"$\lambda$")
    ax3.set_ylim(0.95, 1.25)

    if k < vc.N_MB - 1:                                 # extra axes labelled only on the last panel
        for a in (ax2, ax3):
            a.tick_params(right=False, labelright=False)   # no ticks, no numbers
            a.spines["right"].set_visible(False)           # no extra axis line
    else:
        ax2.set_ylabel(r"$\hat T$ [-]", color="k")
        ax3.set_ylabel(r"$\lambda$ [-]", color="k")

axes[0].set_ylabel(r"$Q_{k,11}$ [MPa]")
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels(); h3, l3 = ax3.get_legend_handles_labels()
fig.legend(h1 + h2 + h3, l1 + l2 + l3, loc="lower center", ncol=5, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))                                                                      # leave the bottom 8 % free for the legend
fig.savefig("test1_all_branches.pdf"); fig.savefig("test1_all_branches_iso.png", dpi=1000)

In [ ]:
branches = [2, 3, 4]                                                                         # indices of branches 3, 4, 5
colors   = ["tab:green", "tab:purple", "tab:brown"]
t_h    = n_ramp * dt_ramp
t_end  = t_h + n_hold * dt_hold
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)

fig, ax = plt.subplots(figsize=(10, 5))
for k, c in zip(branches, colors):
    r = results[k]
    Q_fine = jnp.exp(-t_fine / r["tau_h"]) * r["Q_h"][0, 0]
    ax.plot(t_ramp, r["Q_ramp"][:, 0, 0], "o", color=c, ms=5)
    ax.plot(t_h + t_hold[::5], r["Q_hold"][::5, 0, 0], "s", mfc="none", color=c, ms=5)
    ax.plot(t_h + t_fine, Q_fine, "-", color=c, lw=1.2,
            label=rf"Branch {k+1}, $\tau_k = {float(r['tau_h']):.3g}$ s")
ax.axvline(t_h, color="k", ls="--", lw=1.0)
ax.set_xlabel(r"Time $t$ [s]")
ax.set_ylabel(r"$Q_{k,11}$ [MPa]")
ax.grid(True, alpha=0.3)

ax.plot([], [], "ko", ms=5, label="Ramp (numerical)")                                       # legend entries that
ax.plot([], [], "ks", mfc="none", ms=5, label="Hold (numerical)")                           # explain the markers
ax.plot([], [], "k-", lw=1.2, label="Exact")

ax2 = ax.twinx()
ax2.plot([0.0, t_end], [T_hat, T_hat], color="tab:orange", label=r"$\hat T$")
ax2.set_ylim(-1.0, 1.0)
ax2.set_ylabel(r"$\hat T$ [-]")

ax3 = ax.twinx()                                                                            # 3rd y-axis: stretch
ax3.spines["right"].set_position(("axes", 1.12))                                            # move its axis line further right
ax3.plot(t_lam, lam_t, color="tab:gray", ls="--", lw=1.0, label=r"$\lambda$")
ax3.set_ylim(0.95, 1.25)
ax3.set_ylabel(r"$\lambda$ [-]", color="k")

axes[0].set_ylabel(r"$Q_{k,11}$ [MPa]")
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels(); h3, l3 = ax3.get_legend_handles_labels()
fig.legend(h1 + h2 + h3, l1 + l2 + l3, loc="lower center", ncol=5, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)) 
fig.savefig("test1_branches_345.pdf"); fig.savefig("test1_branches_345.png", dpi=500)

## The $\tau$ map:

In [ ]:
# ---- Grids ----
lambda_grid = jnp.linspace(0.8, 1.5, 71)          # stretch, step 0.01
T_hat_grid  = jnp.linspace(-1.0, 1.0, 41)         # normalized temperature, step 0.05
print("Shapes of the grids:", lambda_grid.shape, T_hat_grid.shape)

# ---- Indices of the points we need ----
i_lam1  = jnp.argmin(jnp.abs(lambda_grid - 1.0))  # reference stretch lambda = 1.0
i_lam12 = jnp.argmin(jnp.abs(lambda_grid - 1.2))  # stretch of test 1
i_T0    = jnp.argmin(jnp.abs(T_hat_grid - 0.0))   # reference temperature T_hat = 0
print(f"lambda = 1.0 at index {int(i_lam1)}, lambda = 1.2 at index {int(i_lam12)}, T_hat = 0 at index {int(i_T0)}")
print("-----------------------------------------------------")

# ---- tau at ONE point (lambda, T_hat) ----
def tau_at(tau_p, T_k, lam, T_hat):
    F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
    C_bar, _ = vc.isochoric_C(vc.right_cauchy_green(F))
    return vc.tau_k(tau_p, T_k, C_bar, T_hat)

k = 2                                             # branch 3
tau_point = tau_at(tau_params[k], vc.T_K[k], 1.2, 0.0)
print("Single point (1.2, 0) matches tau_hold:", bool(jnp.allclose(tau_point, tau_hold)), float(tau_point))
print("-----------------------------------------------------")

# ---- One ROW: fixed lambda, all T_hat (vmap over the 4th argument) ----
tau_row = jax.vmap(tau_at, in_axes=(None, None, None, 0))
row_12 = tau_row(tau_params[k], vc.T_K[k], 1.2, T_hat_grid)                 # shape (41,)

# ---- One COLUMN: fixed T_hat, all lambda (vmap over the 3rd argument) ----
tau_col = jax.vmap(tau_at, in_axes=(None, None, 0, None))
col_0 = tau_col(tau_params[k], vc.T_K[k], lambda_grid, 0.0)                 # shape (71,)

# ---- Step 2, the full MAP: map the ROW function over lambda (nested vmap) ----
tau_map_fn = jax.vmap(tau_row, in_axes=(None, None, 0, None))
tau_map = tau_map_fn(tau_params[k], vc.T_K[k], lambda_grid, T_hat_grid)     # shape (71, 41)
print("Shape of the map:", tau_map.shape)

# ---- Checks: the map must contain the row, the column and the single point ----
print("Map row at lambda = 1.2 equals the row:  ", bool(jnp.allclose(tau_map[i_lam12, :], row_12)))
print("Map column at T_hat = 0 equals the column:", bool(jnp.allclose(tau_map[:, i_T0], col_0)))
print("Map at (1.2, 0) matches tau_hold:        ", bool(jnp.allclose(tau_map[i_lam12, i_T0], tau_hold)))
print("-----------------------------------------------------")

# ---- Relative change w.r.t. the reference (lambda = 1, T_hat = 0), in % ----
tau_ref = tau_map[i_lam1, i_T0]
Z = 100.0 * (tau_map / tau_ref - 1.0)
print(f"Reference tau(1, 0) = {float(tau_ref):.6f} s;  Z at the reference = {float(Z[i_lam1, i_T0])}")
print(f"Z ranges from {float(Z.min()):.4f} % to {float(Z.max()):.4f} %")

In [ ]:
TT, LL = jnp.meshgrid(T_hat_grid, lambda_grid)            # both (71, 41), like Z
m = float(jnp.abs(Z).max())                               # symmetric color range around 0

plt.rcParams.update({"font.size": 13})
fig = plt.figure(figsize=(13, 5))

# ---- left: heat map ----
ax1 = fig.add_subplot(1, 2, 1)
hm = ax1.pcolormesh(T_hat_grid, lambda_grid, Z, shading="auto", cmap="RdBu", vmin=-m, vmax=m)
cb = fig.colorbar(hm, ax=ax1, label=r"$\tau_k / \tau_k(1, 0) - 1$ [%]")
cb.ax.text(0.5, 1.02, "Slower", transform=cb.ax.transAxes, ha="center", va="bottom", fontsize=10)
cb.ax.text(0.5, -0.02, "Faster", transform=cb.ax.transAxes, ha="center", va="top", fontsize=10)
ax1.plot(0.0, 1.0, "k+", ms=12, mew=2,
         label=rf"Reference: $\tau_k(\lambda = 1, \hat T = 0) = {float(tau_ref):.4f}$ s")
# ax1.axhline(1.2, color="k", ls=":", lw=1, label=r"Test 1: $\lambda = 1.2$")
ax1.set_xlabel(r"$\hat T$ [-]"); ax1.set_ylabel(r"$\lambda$ [-]")
ax1.set_title(rf"$\tau_3(\bar{{\mathbf{{C}}}}(\lambda), \hat T)$")
ax1.legend(fontsize=12, loc="upper right")

# ---- right: surface, with the heat map as a shadow on the floor ----
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
floor = float(Z.min()) - 0.5 * m
ax2.plot_surface(TT, LL, Z, cmap="RdBu", vmin=-m, vmax=m, linewidth=0)
ax2.contourf(TT, LL, Z, zdir="z", offset=floor, cmap="RdBu", vmin=-m, vmax=m, alpha=0.6)
ax2.set_zlim(floor, float(Z.max()))
ax2.set_xlabel(r"$\hat T$ [-]"); ax2.set_ylabel(r"$\lambda$ [-]"); ax2.set_zlabel(r"$\tau_k / \tau_k(1, 0) - 1$ [%]")
ax2.set_title(rf"$\tau_3(\bar{{\mathbf{{C}}}}(\lambda), \hat T)$")
ax2.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])              # fewer ticks: no overlapping labels
ax2.view_init(elev=25, azim=-60)

fig.tight_layout()
fig.savefig("tau_map_branch3.pdf"); fig.savefig("tau_map_branch3.png", dpi=1000)

## What the $\tau$ map shows

**What is plotted.** For branch $k = 3$: the relaxation time $\tau_k\big(\bar{\mathbf C}(\lambda), \hat T\big)$ over a grid of uniaxial stretches $\lambda \in [0.8, 1.5]$ and normalized temperatures $\hat T \in [-1, 1]$, shown as the relative change with respect to the reference state:

$$
Z = \frac{\tau_k(\lambda, \hat T)}{\tau_k(1, 0)} - 1 \quad [\%], \qquad \tau_k(1, 0) = 0.5920\ \text{s}
$$

Each point is an independent evaluation of the $\tau$ network at one state: no time, no history.

**How to read it** (red = faster relaxation, blue = slower):

- **Temperature:** $\tau_k$ decreases from left to right, i.e. the branch relaxes faster when heated.
- **Stretch:** both tension ($\lambda > 1$) and compression ($\lambda < 1$) increase $\tau_k$, forming a valley along $\lambda = 1$.
- **Small-strain symmetry (structural, holds for any trained network too):** the network sees deformation only through $\bar I_1 - 3 \approx 3(\lambda - 1)^2$ and $\bar I_2 - 3$, both quadratic in the strain near the reference. Hence $\tau_k$ cannot distinguish tension from compression at small strains; asymmetry appears only at larger strains.
- **Iso-$\tau$ lines** (e.g. the white band) connect states with the same relaxation time. Their slope says how much temperature compensates for how much stretch:
  $\left.\mathrm d\hat T/\mathrm d\lambda\right|_{\tau_k} = -\,(\partial\tau_k/\partial\lambda)\,/\,(\partial\tau_k/\partial\hat T)$.
  Here, a larger stretch needs a higher temperature to relax equally fast.

**Comparison with existing models:**

- **Prony + WLF:** $\tau_k(T) = a_T(T)\,T_k$. No dependence on stretch, and one shift factor $a_T$ shared by all branches. Its map would be identical in every row (vertical stripes), with the same pattern for every branch.
- **vCANN (Abdolazizi, Linka et al.):** $\tau$ is a network of the invariants and of features $\mathbf f$, which can include temperature. The same map could be computed from their model, but $T$ enters as a constant per test in an isothermal theory, learned from a few discrete temperatures.
- **This model:** $T$ is a thermodynamic state variable. $\tau_k(\bar{\mathbf C}, T)$ is evaluated along temperature histories $T(t)$ inside a thermodynamically consistent update, so non-isothermal tests can be used for training and validation, with dense temperature coverage.

**Caveats:**

- The networks are **untrained** (random weights). The shape and the direction of the trends are chance; only the tool and the structural small-strain symmetry are meaningful.
- The variations are **tiny**: about $\pm 0.1\,\%$ over the whole map. The colours and the 3D surface exaggerate them; always read the colorbar.
- Real polymers often relax **faster** when stretched. The trained model, not this map, will decide for PET-A.
- The strain dependence of $\tau_k$ may not be **identifiable** from limited data, since $\tau_k$ is only seen through rate effects.

**What this opens, once the networks are trained on PET-A data:**

- **Time–temperature superposition test:** per-branch shifts $\ln a_{T,k} = \ln\tau_k(1, T) - \ln\tau_k(1, T_{\mathrm{ref}})$. If they coincide, TTS holds; if not, the material is thermorheologically complex, which Prony + WLF cannot represent.
- **Strain–temperature trade-off:** shape and slope of the iso-$\tau$ lines, computed with `jax.grad`.
- **Comparison with WLF:** fitting $C_1$ and $C_2$ to the learned shifts.
- **Ablations:** shared shift vs. $\tau_k(T)$ only vs. full $\tau_k(\bar I_1, \bar I_2, T)$, on held-out tests.
- **Application:** the effect of these dependencies on shape fixity and recovery in the mandrel cycle.
- **Experiments needed:** relaxation holds on a strain × temperature matrix, ideally with temperature changes during a hold.

## Test 2: sinusoidal $T(t)$

In [ ]:
# I can reuse this from Test 1 and the rest of the notebook

# T_hat_initial = 0.0
# LAMBDA = 1.2
# n_ramp, dt_ramp = 10, 0.1                                                       # ramp: 1 s
# n_hold, dt_hold = 100, 0.05                                                     # hold: 5 s, constant step
# lams   = 1.0 + (LAMBDA - 1.0) * jnp.arange(1, n_ramp + 1) / n_ramp              # 1.02, ..., 1.20
# t_ramp = dt_ramp * jnp.arange(0, n_ramp + 1)                                    # 0.0, ..., 1.0  (11 values, incl. t = 0)
# t_hold = dt_hold * jnp.arange(1, n_hold + 1)                                    # time since start of hold (100 values)

In [ ]:
T_test2 = jnp.full(n_ramp, T_hat_initial)
print(T_test2)

In [ ]:
def sinusoidal_T(A, t_hold, P):
        return A * jnp.sin(2 * jnp.pi * t_hold/P)

T_sinusoidal = sinusoidal_T(1.0, t_hold, 2.0)
print(T_sinusoidal.shape)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_hold, T_sinusoidal, "o-", ms=3)
ax.set_xlabel(r"Hold time: $t - t_\mathrm{h}$ [s]")
ax.set_ylabel(r"$\hat T$ [-]")
ax.grid(True, alpha=0.3)
fig.tight_layout()


In [ ]:
A, P = 1.0, 2.0

t_all = jnp.concatenate([t_ramp, t_h + t_hold])
print(t_all.shape, t_all[:12], t_all[-1])

# isothermal: T_hat_initial for the whole history
T_iso_all = jnp.full(n_ramp + 1 + n_hold, T_hat_initial)

# non-isothermal: T_hat_initial during the ramp, then the sine during the hold
T_noniso_all = jnp.concatenate([jnp.full(n_ramp + 1, T_hat_initial), sinusoidal_T(A, t_hold, P)])

print(T_iso_all.shape, T_noniso_all.shape)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_all, T_noniso_all, "o-", ms=3)
ax.set_xlabel(r"$t$ [s]")
ax.set_ylabel(r"$\hat T$ [-]")
ax.grid(True, alpha=0.3)
fig.tight_layout()



In [ ]:
T_hold_iso    = jnp.full(n_hold, T_hat)              # isothermal hold
T_hold_noniso = sinusoidal_T(A, t_hold, P)           # oscillating hold, shape (n_hold,)

results = []
for k in range(vc.N_MB):
    H = H_0[k]
    Q_ramp, S_ramp = [H["Q"]], [H["S_bar"]]                                    # include t = 0
    for lam in lams:
        F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
        C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
        H, _ = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_ramp)
        Q_ramp.append(H["Q"]); S_ramp.append(H["S_bar"])
    H_h = H                                                                    # full state at the start of the hold
    Q_h, tau_h = H_h["Q"], H_h["tau"]

    # ---- the same hold twice, from the same start state: only the temperatures differ ----
    Q_hold = {}
    for name, T_hold in (("iso", T_hold_iso), ("noniso", T_hold_noniso)):
        H = H_h
        Qs = []
        for m in range(n_hold):
            H, _ = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hold[m], dt_hold)
            Qs.append(H["Q"])
        Q_hold[name] = jnp.stack(Qs)

    results.append(dict(Q_ramp=jnp.stack(Q_ramp), S_ramp=jnp.stack(S_ramp),
                        Q_hold_iso=Q_hold["iso"], Q_hold_noniso=Q_hold["noniso"],
                        Q_h=Q_h, tau_h=tau_h))

In [ ]:
# The exact solution only exists for the isothermal hold, so we compare Q_hold_iso with it:

for k, r in enumerate(results):
    Q_exact = jnp.exp(-t_hold / r["tau_h"])[:, None, None] * r["Q_h"]
    err = jnp.abs(r["Q_hold_iso"] - Q_exact).max() / jnp.abs(r["Q_h"]).max()
    dQ  = jnp.abs(r["Q_hold_noniso"] - r["Q_hold_iso"]).max() / jnp.abs(r["Q_h"]).max()
    print(f"Branch {k+1}: tau_h = {float(r['tau_h']):.4g} s,  "
          f"test 1 error = {float(err):.2e},  thermal effect = {100*float(dQ):.2f} %")

In [ ]:
t_h    = n_ramp * dt_ramp
t_end  = t_h + n_hold * dt_hold
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)

# loading history lambda(t): 1 at t = 0, the ramp values, then constant during the hold
t_lam = jnp.concatenate([t_ramp, jnp.array([t_end])])
lam_t = jnp.concatenate([jnp.array([1.0]), lams, jnp.array([LAMBDA])])

fig, axes = plt.subplots(1, vc.N_MB, figsize=(4.4 * vc.N_MB, 3.8))
for k, (ax, r) in enumerate(zip(axes, results)):
    Q_fine = jnp.exp(-t_fine / r["tau_h"]) * r["Q_h"][0, 0]
    Q_iso, Q_non = r["Q_hold_iso"][:, 0, 0], r["Q_hold_noniso"][:, 0, 0]        # NEW: both holds
    ax.plot(t_ramp, r["Q_ramp"][:, 0, 0], "o-", color="tab:red", ms=4, label="Ramp (numerical)")
    ax.plot(t_h + t_hold[::5], Q_iso[::5], "s", mfc="none", color="tab:blue", ms=4,
            label="Hold, isothermal (numerical)")
    ax.plot(t_h + t_fine, Q_fine, "tab:blue", lw=1, label="Hold, isothermal (exact)")
    # NEW: oscillating hold, and the thermal effect shaded by sign
    ax.plot(t_h + t_hold, Q_non, "-", color="tab:green", lw=1.2, label=r"Hold, oscillating $\hat T$")
    up = Q_non >= Q_iso
    ax.fill_between(t_h + t_hold, Q_iso, Q_non, where=up, interpolate=True,
                    color="tab:blue", alpha=0.25, label="Above isothermal")
    ax.fill_between(t_h + t_hold, Q_iso, Q_non, where=~up, interpolate=True,
                    color="tab:red", alpha=0.25, label="Below isothermal")
    ax.set_title(rf"Branch {k+1}, $\tau_k = {float(r['tau_h']):.3g}$ s")
    ax.set_xlabel(r"Time $t$ [s]")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()                                    # 2nd y-axis: temperature
    # NEW: both temperature histories (needs t_all, T_iso_all, T_noniso_all)
    ax2.plot(t_all, T_iso_all, color="tab:orange", lw=1.0, label=r"$\hat T$ isothermal")
    ax2.plot(t_all, T_noniso_all, color="tab:orange", ls=":", lw=1.2, label=r"$\hat T$ sinusoidal")
    ax2.set_ylim(-1.0, 1.0)

    ax3 = ax.twinx()                                    # 3rd y-axis: stretch
    ax3.spines["right"].set_position(("axes", 1.22))    # move its axis line further right
    ax3.plot(t_lam, lam_t, color="tab:gray", ls="--", lw=1.0, label=r"$\lambda$")
    ax3.set_ylim(0.95, 1.25)

    if k < vc.N_MB - 1:                                 # extra axes labelled only on the last panel
        for a in (ax2, ax3):
            a.tick_params(right=False, labelright=False)   # no ticks, no numbers
            a.spines["right"].set_visible(False)           # no extra axis line
    else:
        ax2.set_ylabel(r"$\hat T$ [-]", color="k")
        ax3.set_ylabel(r"$\lambda$ [-]", color="k")

axes[0].set_ylabel(r"$Q_{k,11}$ [MPa]")
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels(); h3, l3 = ax3.get_legend_handles_labels()
fig.legend(h1 + h2 + h3, l1 + l2 + l3, loc="lower center", ncol=5, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.12, 1, 1))                  # CHANGED: 12 % for the now two-row legend
fig.savefig("test1_all_branches.pdf"); fig.savefig("test1_all_branches.png", dpi=1000)

The 36% fast-branch difference looks large only because this branch's $\mathbf Q_{k,h}$ is tiny, so the steady oscillation is large relative to it.

## What to study after obtaining this plot:

1. Why the fast branch seems to follow the heating/cooling rate? Is it just the scale?
2. Differentiate two different temperature effects:

    A. Temperature changing $\tau_k$
    
    B. Temperature changing $\bar S_k$

3. Timing: why does each branch respond with different amplitude and phase?

## Isothermal vs. oscillating hold: what the plot does and does not tell

The networks are **untrained**, so the numbers say nothing about PET-A. What matters is which features come from the **model structure** and which from the random weights.

**Structural: will hold for any trained network**

- Isothermal holds relax as exact exponentials, $\mathbf Q_k(t) = \mathbf Q_{k,\mathrm h}\,e^{-(t - t_\mathrm h)/\tau_k}$ (test 1).
- At $\bar{\mathbf C} = \mathbf I$, temperature changes produce no overstress at all, because the normalization gives $\bar{\mathbf S}_k(\mathbf I, T) = \mathbf 0$ for every $T$ (test 2).
- At a stretched state, temperature changes **do** drive $\mathbf Q_k$, through the thermal term of the evolution equation:
  $$
  \dot{\mathbf Q}_k + \frac{\mathbf Q_k}{\tau_k} = \tfrac12\bar{\mathbb C}_k : \dot{\bar{\mathbf C}} + \partial_T\bar{\mathbf S}_k\,\dot T
  $$
- While $T$ keeps changing, $\mathbf Q_k$ does not decay to zero; it oscillates around it (memory of the temperature path).
- Fast branches ($\tau_k \ll P$) follow the **rate**: $\mathbf Q_k \approx \tau_k\,\partial_T\bar{\mathbf S}_k\,\dot T$, with peaks where $\hat T$ crosses zero. Slow branches add the oscillation on top of their decay.

**Chance: decided only by data**

- The **sign** of the thermal effect in each branch, i.e. the sign of $\partial_T\bar{\mathbf S}_k$.
- Its **size** (here about 1–4 % of $\mathbf Q_{k,\mathrm h}$; branch 1 only looks larger because its $\mathbf Q_{k,\mathrm h}$ is tiny).
- The almost flat $\tau_k(T)$: untrained, $\tau_k$ changes by less than 1 % over the whole temperature range.
- Which branch reacts most.

**What to expect for amorphous PET-A** (qualitative expectation from polymer physics, to be checked against data)

- Crossing $T_\mathrm g$, the material softens strongly, so $\partial_T\bar{\mathbf S}_k$ is likely **negative and large** there.
- In the rubbery range above $T_\mathrm g$, entropic elasticity makes the stiffness **increase** with temperature, which could flip the sign of $\partial_T\bar{\mathbf S}_k$.
- $\tau_k$ dropping by decades on heating will dominate the response: instead of gentle wiggles, the branches freeze on cooling and release on heating, the mechanism behind the shape-memory cycle.
- The most direct experimental check: tests with temperature changes during a hold at fixed stretch.



### Caveat: softplus limits how far $\tau_k$ can rise above $T_k$

The output $\tau_k = T_k\,\operatorname{softplus}(\mathcal N_k)$ behaves very differently in each direction:

$$
\operatorname{softplus}(\mathcal N) \approx e^{\mathcal N} \quad (\mathcal N \ll 0), \qquad \operatorname{softplus}(\mathcal N) \approx \mathcal N \quad (\mathcal N \gg 0)
$$

- **Below $T_k$ is easy:** six decades down need only $\mathcal N_k \approx -14$.
- **Above $T_k$ is hard:** six decades up need $\mathcal N_k \approx 10^6$, which is out of reach in practice.

So whether a glassy plateau plus a sharp drop at $T_\mathrm g$ can be represented depends on where the plateau lies relative to $T_k$. If the plateau is near or below $T_k$, the drop is a downward change and softplus handles it. If it lies far above $T_k$, the cold side comes out too soft.

**Possible fixes:** use $\tau_k = T_k\,e^{\mathcal N_k}$, which is symmetric in $\log\tau$; place the $T_k$ near the glassy end; or make $T_k$ learnable.

**Second requirement:** the drop must happen within a narrow $\hat T$ range, so the temperature weights $\mathbf w_T$ must grow large during training.

**Check before training on data:** fit one $\tau$ network to a synthetic $\log_{10}\tau(\hat T)$ with a plateau and a drop of several decades. Do it once with softplus and once with the exponential output, and compare.

In [ ]:
branches = [2, 3, 4]                                                                         # indices of branches 3, 4, 5
colors   = ["tab:green", "tab:purple", "tab:brown"]
t_h    = n_ramp * dt_ramp
t_end  = t_h + n_hold * dt_hold
t_fine = jnp.linspace(0.0, n_hold * dt_hold, 400)

fig, ax = plt.subplots(figsize=(10, 5.5))
for k, c in zip(branches, colors):
    r = results[k]
    Q_iso    = r["Q_hold_iso"][:, 0, 0]
    Q_noniso = r["Q_hold_noniso"][:, 0, 0]
    Q_fine   = jnp.exp(-t_fine / r["tau_h"]) * r["Q_h"][0, 0]
    ax.plot(t_ramp, r["Q_ramp"][:, 0, 0], "o", color=c, ms=5)
    ax.plot(t_h + t_hold[::5], Q_iso[::5], "s", mfc="none", color=c, ms=5)
    ax.plot(t_h + t_fine, Q_fine, "-", color=c, lw=1.2,
            label=rf"Branch {k+1}, $\tau_k = {float(r['tau_h']):.3g}$ s")
    ax.plot(t_h + t_hold, Q_noniso, "-.", color=c, lw=1.2)                                  # oscillating-T hold
    ax.fill_between(t_h + t_hold, Q_iso, Q_noniso, where=Q_noniso >= Q_iso,                 # difference, by sign
                    color="tab:blue", alpha=0.15, interpolate=True, lw=0)
    ax.fill_between(t_h + t_hold, Q_iso, Q_noniso, where=Q_noniso < Q_iso,
                    color="tab:red", alpha=0.15, interpolate=True, lw=0)
ax.axvline(t_h, color="k", ls="--", lw=1.0)
ax.set_xlabel(r"Time $t$ [s]")
ax.set_ylabel(r"$Q_{k,11}$ [MPa]")
ax.grid(True, alpha=0.3)

ax.plot([], [], "ko", ms=5, label="Ramp (numerical)")                                       # legend entries that
ax.plot([], [], "ks", mfc="none", ms=5, label=r"Hold, $\hat T$ const. (numerical)")         # explain the markers
ax.plot([], [], "k-", lw=1.2, label="Exact")
ax.plot([], [], "k-.", lw=1.2, label=r"Hold, $\hat T$ oscillating")
ax.fill_between([], [], [], color="tab:blue", alpha=0.15, label="oscillating > const.")
ax.fill_between([], [], [], color="tab:red", alpha=0.15, label="oscillating < const.")

ax2 = ax.twinx()
ax2.plot(t_all, T_iso_all, color="tab:orange", lw=1.0, label=r"$\hat T$ const.")
ax2.plot(t_all, T_noniso_all, color="tab:orange", ls=":", lw=1.2, label=r"$\hat T$ oscillating")
ax2.set_ylim(-1.1, 1.1)
ax2.set_ylabel(r"$\hat T$ [-]")

ax3 = ax.twinx()                                                                            # 3rd y-axis: stretch
ax3.spines["right"].set_position(("axes", 1.12))                                            # move its axis line further right
ax3.plot(t_lam, lam_t, color="tab:gray", ls="--", lw=1.0, label=r"$\lambda$")
ax3.set_ylim(0.95, 1.25)
ax3.set_ylabel(r"$\lambda$ [-]", color="k")

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels(); h3, l3 = ax3.get_legend_handles_labels()
fig.legend(h1 + h2 + h3, l1 + l2 + l3, loc="lower center", ncol=4, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.16, 1, 1))
fig.savefig("test1_branches_345_noniso.pdf"); fig.savefig("test1_branches_345_noniso.png", dpi=500)

## Bowl-shaped Branch Potential

In [ ]:
W_T_FACTOR = 20.0
psi_params_amplified = [{**p, "w_T": W_T_FACTOR * p["w_T"]} for p in psi_params]
print(psi_params_amplified[2]["w_T"] / psi_params[2]["w_T"])

In [ ]:
def psi_at(psi_p, lambda_1, lambda_2, T_hat):
    C_bar = jnp.diag(jnp.array([lambda_1**2, lambda_2**2, 1/(lambda_1**2*lambda_2**2)]))
    I1_bar, I2_bar = vc.invariants(C_bar)
    potential_k = vc.psi_corrected(psi_p, C_bar, T_hat)
    
    return (I1_bar, I2_bar, potential_k)

# Checking correcting initial output of the network
for T in [-1.0, 0.0, 1.0]:
    print(T, psi_at(psi_params_amplified[2], 1.0, 1.0, T))

print("------------------------------------------------------------------------------------------")

lam = 1.2
for T in [-1.0, 0.0, 1.0]:
    print(T, psi_at(psi_params_amplified[2], lam, 1/jnp.sqrt(lam), T))

print("------------------------------------------------------------------------------------------")

In [ ]:
lambda_1_grid = jnp.linspace(0.7, 1.4, 61)
lambda_2_grid = lambda_1_grid

# tau_map_fn = jax.vmap(tau_row, in_axes=(None, None, 0, None))
# tau_map = tau_map_fn(tau_params[k], vc.T_K[k], lambda_grid, T_hat_grid)     # shape (71, 41)

psi_row = jax.vmap(psi_at, in_axes = (None, None, 0, None))                   # replaces the INNER loop: maps over l2 (3rd argument)
psi_grid = jax.vmap(psi_row, in_axes=(None, 0, None, None))                   # replaces the INNER loop: maps over l1 (2nd argument)

I1, I2, Psi = psi_grid(psi_params_amplified[2], lambda_1_grid, lambda_2_grid, T_hat)            # each of shape (len(l1_grid), len(l2_grid))

In [ ]:
# --- Cell A: grid over strain size r and deformation mode theta ---
K = 2
p_amp = psi_params_amplified[K]
LAM_MAX = 1.4                                            # largest uniaxial stretch shown
r_grid     = jnp.linspace(0.0, jnp.log(LAM_MAX) / jnp.sqrt(2/3), 41)   # strain size (log strain)
theta_grid = jnp.linspace(0.0, 2 * jnp.pi, 73)          # full turn: tension at 0/120/240 deg, compression at 60/180/300 deg
R, TH = jnp.meshgrid(r_grid, theta_grid, indexing="ij")  # (41, 73)
L1 = jnp.exp(R * jnp.sqrt(2/3) * jnp.cos(TH))                     # principal stretches from log strains
L2 = jnp.exp(R * jnp.sqrt(2/3) * jnp.cos(TH - 2 * jnp.pi / 3))    # (lambda_3 follows from det = 1 inside psi_at)

psi_pairs = jax.vmap(psi_at, in_axes=(None, 0, 0, None))          # maps over (l1, l2) pairs
def psi_wedge(p, T_hat):
    I1, I2, Psi = psi_pairs(p, L1.ravel(), L2.ravel(), T_hat)
    shape = L1.shape
    return I1.reshape(shape), I2.reshape(shape), Psi.reshape(shape)

T_slices = [-1.0, 0.0, 1.0]
grids = {T: psi_wedge(p_amp, T) for T in T_slices}
for T, (I1, I2, Psi) in grids.items():                  # r = 0 row is the reference state
    print(f"T_hat={T:+.0f}: at r=0: I1={I1[0,0]:.6f}, I2={I2[0,0]:.6f}, Psi={Psi[0,0]:.1e};  "
          f"tension edge end: I1={I1[-1,0]:.4f} (expect {LAM_MAX**2 + 2/LAM_MAX:.4f}); "
          f"symmetry 0 vs 120 deg: {float(Psi[-1,0]):.6f} vs {float(Psi[-1,24]):.6f}")

# plot coordinates: strain size r and mode theta as polar coordinates (the "deviatoric plane")
X, Y = R * jnp.cos(TH), R * jnp.sin(TH)
i_ten = jnp.argmin(jnp.abs(theta_grid - 0.0))            # theta = 0      -> uniaxial tension (your data)
i_com = jnp.argmin(jnp.abs(theta_grid - jnp.pi / 3))     # theta = 60 deg -> uniaxial compression

In [ ]:
# --- Cell B: static figure, three temperatures, same z-limits ---
z_max = max(float(Psi.max()) for (_, _, Psi) in grids.values())
fig = plt.figure(figsize=(16, 5.2))
for n, T in enumerate(T_slices):
    I1, I2, Psi = grids[T]
    ax = fig.add_subplot(1, 3, n + 1, projection="3d")
    ax.computed_zorder = False                          # draw in the order given: rays on top of the surface
    ax.plot_surface(X, Y, Psi, cmap="viridis", vmin=0.0, vmax=z_max,
                    rstride=1, cstride=1, linewidth=0.15, edgecolor="k", alpha=0.75, zorder=1)
    ax.plot(X[:, i_ten], Y[:, i_ten], Psi[:, i_ten], color="tab:red",  lw=3, zorder=10, label="uniaxial tension")
    ax.plot(X[:, i_com], Y[:, i_com], Psi[:, i_com], color="tab:blue", lw=3, zorder=10, label="uniaxial compression")
    ax.scatter([0.0], [0.0], [0.0], color="k", s=30, label="reference state", zorder=11)
    ax.set_zlim(0.0, z_max)
    ax.set_xlabel(r"$r\cos\theta$"); ax.set_ylabel(r"$r\sin\theta$")
    ax.set_zlabel(r"$\Psi_k$")
    ax.set_title(rf"$\hat T = {T:+.0f}$")
    ax.view_init(elev=30, azim=-55)
    if n == 0: ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig("psi_bowl_branch3_amplified.png", dpi=200)

In [ ]:
# --- Cell C (with shadow): animation over a temperature history T_hat(t) ---

from matplotlib.animation import FuncAnimation 

n_frames = 100
t_anim = jnp.linspace(0.0, 1.0, n_frames)
T_anim = jnp.cos(2 * jnp.pi * t_anim)                    # +1 (hot) -> -1 (cold) -> +1 (hot)
frames = [psi_wedge(p_amp, T) for T in T_anim]
z_max_anim = max(float(f[2].max()) for f in frames)
shadow_levels = jnp.linspace(0.0, z_max_anim, 41)        # fill levels, fixed for all frames
line_levels   = jnp.linspace(0.0, z_max_anim, 9)[1:]     # contour lines, fixed: watch them move inward/outward
rim = (X[-1, :], Y[-1, :], jnp.zeros_like(X[-1, :]))     # external contour (fixed footprint)

fig_a = plt.figure(figsize=(7.5, 6.5))
ax_a = fig_a.add_subplot(projection="3d")
def draw(f):
    ax_a.clear(); ax_a.computed_zorder = False
    I1, I2, Psi = frames[f]
    # shadow on the Psi = 0 floor
    ax_a.contourf(X, Y, Psi, levels=shadow_levels, zdir="z", offset=0.0, cmap="viridis", alpha=0.15, zorder=0)
    ax_a.contour(X, Y, Psi, levels=line_levels, zdir="z", offset=0.0, colors="k", linewidths=0.7, alpha=0.6, zorder=1)
    ax_a.plot(*rim, color="k", lw=1.0, zorder=2)
    # bowl and rays
    ax_a.plot_surface(X, Y, Psi, cmap="viridis", vmin=0.0, vmax=z_max_anim,
                      rstride=1, cstride=1, linewidth=0.15, edgecolor="k", alpha=0.75, zorder=3)
    ax_a.plot(X[:, i_ten], Y[:, i_ten], Psi[:, i_ten], color="tab:red",  lw=3, zorder=10)
    ax_a.plot(X[:, i_com], Y[:, i_com], Psi[:, i_com], color="tab:blue", lw=3, zorder=10)
    ax_a.set_zlim(0.0, z_max_anim)
    ax_a.set_xlabel(r"$r\cos\theta$"); ax_a.set_ylabel(r"$r\sin\theta$"); ax_a.set_zlabel(r"$\Psi_k$")
    ax_a.set_title(rf"$t$ = {float(t_anim[f]):.2f},  $\hat T$ = {float(T_anim[f]):+.2f}")
    ax_a.view_init(elev=25, azim=-55)
anim = FuncAnimation(fig_a, draw, frames=n_frames, interval=100)
anim.save("psi_bowl_branch3_amplified_shadow.gif", writer="pillow", fps=10)
anim.save("psi_bowl_branch3_amplified_shadow.mp4", writer="ffmpeg", fps=10, dpi=500)
# In Jupyter: from IPython.display import HTML; HTML(anim.to_jshtml())

In [ ]:
# --- Cell D: the bowl's shadow on the Psi = 0 plane ---
T_show = 0.0
I1, I2, Psi = grids[T_show]
levels = jnp.linspace(0.0, float(Psi.max()), 9)[1:]      # 8 energy levels above the bottom

fig = plt.figure(figsize=(14, 6.6))

# left: bowl with its contour lines projected onto the floor (Psi = 0)
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.computed_zorder = False
ax.contourf(X, Y, Psi, levels=jnp.linspace(0.0, float(Psi.max()), 41), zdir="z", offset=0.0,
            cmap="viridis", alpha=0.15, zorder=0)                       # filled shadow
ax.contour(X, Y, Psi, levels=levels, zdir="z", offset=0.0, colors="k", linewidths=0.8, zorder=1)
ax.plot(X[-1, :], Y[-1, :], jnp.zeros_like(X[-1, :]), color="k", lw=1.0, zorder=2)   # external contour (rim)
ax.plot_surface(X, Y, Psi, cmap="viridis", rstride=1, cstride=1, linewidth=0.15,
                edgecolor="k", alpha=0.55, zorder=3)
ax.plot(X[:, i_ten], Y[:, i_ten], Psi[:, i_ten], color="tab:red",  lw=3, zorder=10)
ax.plot(X[:, i_com], Y[:, i_com], Psi[:, i_com], color="tab:blue", lw=3, zorder=10)
ax.set_zlim(0.0, float(Psi.max()))
ax.set_xlabel(r"$r\cos\theta$"); ax.set_ylabel(r"$r\sin\theta$"); ax.set_zlabel(r"$\Psi_k$")
ax.set_title(rf"Branch {K+1}, $\hat T = {T_show:+.0f}$: bowl and its shadow")
ax.view_init(elev=22, azim=-55)

# right: the shadow seen from above (top view of the floor)
ax2 = fig.add_subplot(1, 2, 2)
cf = ax2.contourf(X, Y, Psi, levels=41, cmap="viridis")
cs = ax2.contour(X, Y, Psi, levels=levels, colors="k", linewidths=0.8)
ax2.clabel(cs, fmt="%.1f", fontsize=8)
ax2.plot(X[-1, :], Y[-1, :], color="k", lw=2.0, label=rf"external contour: $\lambda$ = {LAM_MAX} in tension")
ax2.plot(X[:, i_ten], Y[:, i_ten], color="tab:red",  lw=3, label="uniaxial tension (your data)")
ax2.plot(X[:, i_com], Y[:, i_com], color="tab:blue", lw=3, label="uniaxial compression")
ax2.plot(0, 0, "ko", ms=6)
ax2.set_aspect("equal")
ax2.set_xlabel(r"$r\cos\theta$"); ax2.set_ylabel(r"$r\sin\theta$")
ax2.set_title(r"Top view: lines of constant $\Psi_k$")
ax2.legend(loc="lower left", fontsize=8, framealpha=0.9)
fig.colorbar(cf, ax=ax2, label=r"$\Psi_k$", shrink=0.85)
fig.tight_layout()
fig.savefig("psi_bowl_shadow_branch3.png", dpi=200)

In [ ]:
# --- Cell E: steeper only, or also a different shape? (profiles along the tension ray) ---
T_list = [-1.0, -0.5, 0.0, 0.5, 1.0]
rr = R[:, i_ten]                                           # strain size along the tension ray
prof = {T: psi_wedge(p_amp, T)[2][:, i_ten] for T in T_list}

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 4.6))
cmap = plt.get_cmap("coolwarm")
for j, T in enumerate(T_list):
    c = cmap(j / (len(T_list) - 1))
    a1.plot(rr, prof[T], color=c, lw=2, label=rf"$\hat T = {T:+.1f}$")
    a2.plot(rr, prof[T] / prof[T][-1], color=c, lw=2)
a1.set_xlabel(r"strain size $r$"); a1.set_ylabel(r"$\Psi_k$ along tension")
a1.set_title("(a) Same footprint, different height"); a1.legend(fontsize=9); a1.grid(alpha=0.3)
a2.set_xlabel(r"strain size $r$"); a2.set_ylabel(r"$\Psi_k / \Psi_k(r_{\max})$")
a2.set_title("(b) Normalized: do the profiles collapse?"); a2.grid(alpha=0.3)

# (c) where does a fixed energy level sit? radius of the contour Psi = level, vs temperature
T_fine = jnp.linspace(-1.0, 1.0, 41)
for level in [1.0, 2.0, 4.0]:
    radii = [float(jnp.interp(level, psi_wedge(p_amp, T)[2][:, i_ten], rr)) for T in T_fine]
    a3.plot(T_fine, radii, lw=2, label=rf"contour $\Psi_k$ = {level:g}")
a3.set_xlabel(r"$\hat T$"); a3.set_ylabel(r"radius $r$ of the contour")
a3.set_title("(c) A fixed energy level moves inward when steeper"); a3.legend(fontsize=9); a3.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("psi_steeper_check_branch3.png", dpi=200)

# numbers: curvature at the bottom (~ mu) and shape ratio
for T in T_list:
    p_ = prof[T]; mu_eff = float(p_[1] / rr[1]**2)       # Psi ~ mu r^2 near the bottom
    print(f"T_hat={T:+.1f}: Psi(r_max)={float(p_[-1]):.3f}, mu_eff at bottom={mu_eff:.2f}, "
          f"Psi(r_max/2)/Psi(r_max)={float(p_[len(p_)//2]/p_[-1]):.4f}  (pure r^2 gives 0.25)")

## Test 3: Sign check. Pressure-free uniaxial

### Three stresses in a branch

| Symbol | What it is | Depends on |
|---|---|---|
| $\bar{\mathbf S}_k = 2\,\partial\Psi_k/\partial\bar{\mathbf C}$ | The **generating stress**: what the branch would carry if its dashpot were locked from the start. It drives the update through its increments $\bar{\mathbf S}_{k,n+1} - \bar{\mathbf S}_{k,n}$. It is the slope of the $\Psi_k$ bowl at the current point, its gradient **. | Current state only |
| $\mathbf Q_k$ | The **overstress**, the internal variable: what the spring actually holds and the dashpot feels. Its energy is $\Psi_k^{\mathrm{neq}} = \mathbf Q_k:\mathbf Q_k/(4\mu_k)$. | History |
| $\bar{\mathbf S}_k^{\mathrm{neq}} = \frac{1}{2\mu_k}\,\bar{\mathbb C}_k:\mathbf Q_k$ | The branch's **contribution to the fictitious stress**: $\mathbf Q_k$ mapped through the branch tangent. This is what `bu` returns as its second output. | History |

** Remember the slope lives in more dimensions than the plot:

$$
\frac{\mathrm d\Psi_k}{\mathrm dr} = \frac{\partial\Psi_k}{\partial\bar{\mathbf C}} : \frac{\mathrm d\bar{\mathbf C}}{\mathrm dr} = \tfrac12\,\bar{\mathbf S}_k : \frac{\mathrm d\bar{\mathbf C}}{\mathrm dr}
$$

In [ ]:
print(lams)

In [ ]:
results_test3 = []
for k in range(vc.N_MB):
    
    H = H_0[k]
    Q_ramp, S_ramp = [H["Q"]], [H["S_bar"]]                                                                     # include t = 0
    S_neq_ramp = [jnp.zeros((3,3))]                                                                               # initialize isochoric stress tensor
    
    for lam in lams:
        F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
        C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
        H, S_neq = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_ramp)
        Q_ramp.append(H["Q"]); S_ramp.append(H["S_bar"]); S_neq_ramp.append(S_neq)
    
    H_h = H                                                                                                     # full state at the start of the hold
    Q_h, tau_h = H_h["Q"], H_h["tau"]

    # ---- the same hold twice, from the same start state: only the temperatures differ ----
    Q_hold, S_neq_hold = {}, {}

    for name, T_hold in (("iso", T_hold_iso), ("noniso", T_hold_noniso)):
        H_hold = H_h
        Qs, Ss = [], []
        for m in range(n_hold):
            H_hold, S_neq = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H_hold, C_bar, T_hold[m], dt_hold)
            Qs.append(H_hold["Q"]); Ss.append(S_neq)
        
        Q_hold[name] = jnp.stack(Qs)
        S_neq_hold[name] = jnp.stack(Ss)

    results_test3.append(dict(Q_ramp=jnp.stack(Q_ramp), S_ramp=jnp.stack(S_ramp), S_neq_ramp=jnp.stack(S_neq_ramp),
                        Q_hold_iso=Q_hold["iso"], Q_hold_noniso=Q_hold["noniso"],
                        S_neq_hold_iso = S_neq_hold["iso"], S_neq_hold_noniso = S_neq_hold["noniso"],
                        Q_h=Q_h, tau_h=tau_h))

print("Elements and shape in the result's list:")
r = results_test3[2]                                   # branch 3
{name: v.shape for name, v in r.items()}               # every key with its shape

      

$$
d_k = \lambda^2\,\bar S^{\mathrm{neq}}_{k,11} - \lambda^{-1}\,\bar S^{\mathrm{neq}}_{k,22} = \sigma^{\mathrm{neq}}_{k,11} - \sigma^{\mathrm{neq}}_{k,22}
$$

In [ ]:
print("---------------------------------------------------------------------------")
print("Quick check of tensor & components:")
print("Lambda:", lams[-1], lams.shape)
print("Last entry of the ramp stress tensor:", results_test3[2]["S_neq_ramp"][-1])
print("S11 component:", r["S_neq_ramp"][-1, 0, 0])
print("S22 component:", r["S_neq_ramp"][-1, 1, 1])
print("---------------------------------------------------------------------------")

def dk(lam, S_bar_k_neq_11, S_bar_k_neq_22):
    return lam**2*S_bar_k_neq_11 - lam**-1*S_bar_k_neq_22

difference_k_ramp, difference_k_hold = [], []
for k in range(vc.N_MB):
    diff_k_ramp = dk(lams, results_test3[k]["S_neq_ramp"][1:, 0, 0], results_test3[k]["S_neq_ramp"][1:, 1, 1])
    diff_k_hold = dk(lams[-1], results_test3[k]["S_neq_hold_iso"][:, 0, 0], results_test3[k]["S_neq_hold_iso"][:, 1, 1] )
    difference_k_ramp.append(diff_k_ramp); difference_k_hold.append(diff_k_hold)

diff_k_ramp, diff_k_hold=jnp.array(difference_k_ramp), jnp.array(difference_k_hold)
if jnp.all(diff_k_ramp > 0):
    print("Every difference is >0 for the ramp.")
else:
    print("Something is wrong at the ramp:", jnp.all(diff_k_ramp > 0, axis=1))
if jnp.all(diff_k_hold > 0):
    print("Every difference is >0 for the hold.")
else:
    print("Something is wrong at the hold:", jnp.all(diff_k_hold > 0, axis=1))

decay_k_hold = jnp.diff(diff_k_hold, axis=1)          # (5, 99): change from one hold step to the next
TOL = 1e-12                                            # round-off allowance once a branch has relaxed to ~0

if jnp.all(decay_k_hold < TOL):
    print("The difference decays at every hold step for every branch.")
else:
    print("Something is wrong in the decay:", jnp.all(decay_k_hold < TOL, axis=1))
    print("Largest increase per branch:", jnp.max(decay_k_hold, axis=1))



In [ ]:
# ---- compression: same ramp + hold, lambda -> 0.8 ----
LAMBDA_C = 0.8
lams_c = 1.0 + (LAMBDA_C - 1.0) * jnp.arange(1, n_ramp + 1) / n_ramp       # 0.98, ..., 0.80

results_test3_c = []
for k in range(vc.N_MB):
    H = H_0[k]
    Q_ramp, S_ramp = [H["Q"]], [H["S_bar"]]
    S_neq_ramp = [jnp.zeros((3,3))]
    for lam in lams_c:
        F = jnp.diag(jnp.array([lam, lam**-0.5, lam**-0.5]))
        C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
        H, S_neq = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hat, dt_ramp)
        Q_ramp.append(H["Q"]); S_ramp.append(H["S_bar"]); S_neq_ramp.append(S_neq)
    H_h = H

    H_hold = H_h
    Qs, Ss = [], []
    for m in range(n_hold):
        H_hold, S_neq = bu(psi_params[k], tau_params[k], vc.T_K[k], MU_K, H_hold, C_bar, T_hold_iso[m], dt_hold)
        Qs.append(H_hold["Q"]); Ss.append(S_neq)

    results_test3_c.append(dict(Q_ramp=jnp.stack(Q_ramp), S_ramp=jnp.stack(S_ramp), S_neq_ramp=jnp.stack(S_neq_ramp),
                                Q_hold_iso=jnp.stack(Qs), S_neq_hold_iso=jnp.stack(Ss)))

# ---- sign check: everything negative, relaxing towards 0 ----
difference_k_ramp_c, difference_k_hold_c = [], []
for k in range(vc.N_MB):
    diff_k_ramp_c = dk(lams_c, results_test3_c[k]["S_neq_ramp"][1:, 0, 0], results_test3_c[k]["S_neq_ramp"][1:, 1, 1])
    diff_k_hold_c = dk(lams_c[-1], results_test3_c[k]["S_neq_hold_iso"][:, 0, 0], results_test3_c[k]["S_neq_hold_iso"][:, 1, 1])
    difference_k_ramp_c.append(diff_k_ramp_c); difference_k_hold_c.append(diff_k_hold_c)

diff_k_ramp_c, diff_k_hold_c = jnp.array(difference_k_ramp_c), jnp.array(difference_k_hold_c)
if jnp.all(diff_k_ramp_c < 0):
    print("Every difference is <0 for the compression ramp.")
else:
    print("Something is wrong at the compression ramp:", jnp.all(diff_k_ramp_c < 0, axis=1))
if jnp.all(diff_k_hold_c < 0):
    print("Every difference is <0 for the compression hold.")
else:
    print("Something is wrong at the compression hold:", jnp.all(diff_k_hold_c < 0, axis=1))

decay_k_hold_c = jnp.diff(diff_k_hold_c, axis=1)      # negative values relaxing towards 0 -> increasing
if jnp.all(decay_k_hold_c > -TOL):
    print("The difference relaxes towards 0 at every hold step for every branch.")
else:
    print("Something is wrong in the compression decay:", jnp.all(decay_k_hold_c > -TOL, axis=1))
    print("Largest decrease per branch:", jnp.min(decay_k_hold_c, axis=1))

print("End of compression ramp, per branch:", diff_k_ramp_c[:, -1])

In [ ]:
print(len(results_test3))
for k, x in enumerate(results_test3):
    print(k + 1,
          "Q11 end of ramp:", float(x["Q_ramp"][-1, 0, 0]),
          "| max |S_neq| over ramp:", float(jnp.abs(x["S_neq_ramp"]).max()))

## Test 4: Free heating at the reference state.

In [ ]:
results_test4_freeheating = []
H_0_amp = [initial_history(psi_params_amplified[k], tau_params[k], vc.T_K[k], jnp.eye(3), T_hat_initial) for k in range(vc.N_MB)]

for k in range(vc.N_MB):
    H = H_0_amp[k]
    Q_null, S_null = [H["Q"]], [H["S_bar"]]                                                                             # include t = 0
    S_neq_null = [jnp.zeros((3,3))]                                                                                     # initialize

    for m in range(T_hold_noniso.shape[0]):
        F=jnp.diag(jnp.array([1.0, 1.0, 1.0]))                                                                          # no deformation
        C_bar, J = vc.isochoric_C(vc.right_cauchy_green(F))
        H, S_neq = bu(psi_params_amplified[k], tau_params[k], vc.T_K[k], MU_K, H, C_bar, T_hold_noniso[m], dt_hold)
        Q_null.append(H["Q"]); S_null.append(H["S_bar"]); S_neq_null.append(S_neq)

    results_test4_freeheating.append(dict(Q_null=jnp.stack(Q_null), S_null=jnp.stack(S_null), S_neq_null=jnp.stack(S_neq_null)))

# Testing the null deformation: the stress should remain zero, and the internal state should not change
print("---------------------------------------------------------------------------")
Q_max = jnp.array([jnp.abs(r["Q_null"]).max() for r in results_test4_freeheating])
S_max = jnp.array([jnp.abs(r["S_neq_null"]).max() for r in results_test4_freeheating])
print("max |Q| per branch:    ", Q_max)
print("max |S_neq| per branch:", S_max)
if jnp.all(Q_max < 1e-12) and jnp.all(S_max < 1e-12):
    print("Free heating: no overstress and no branch stress in any branch.")
else:
    print("Free heating FAILED for branches:", [k + 1 for k in range(vc.N_MB) if Q_max[k] >= 1e-12 or S_max[k] >= 1e-12])

print("---------------------------------------------------------------------------")
print(
    "max |S_bar_k(I,T_hat)| per branch:", jnp.array([jnp.abs(r["S_null"]).max() for r in results_test4_freeheating]))
print("---------------------------------------------------------------------------")

## Plotting $\partial\Psi/\partial{\bar C}$ and $\partial/\partial{\bar C}(\partial\Psi/\partial{\bar C})$

In [ ]:
import jax.numpy as jnp, numpy as np, matplotlib; matplotlib.use("Agg")
plt.rcParams.update({"mathtext.fontset": "cm", "font.size": 10})

key_psi, _ = jax.random.split(jax.random.PRNGKey(0))
psi_params = [vc.init_psi_params(k) for k in jax.random.split(key_psi, vc.N_MB)]
p_amp = {**psi_params[2], "w_T": 20.0 * psi_params[2]["w_T"]}
T_HAT = 0.0

# bowl coordinates: x = r cos(theta), y = r sin(theta); log strains e_i = sqrt(2/3) (x cos(2 pi i/3) + y sin(2 pi i/3))
def psi_xy(xy, T_hat):
    a = 2*jnp.pi*jnp.arange(3)/3
    e = jnp.sqrt(2/3)*(xy[0]*jnp.cos(a) + xy[1]*jnp.sin(a))
    return vc.psi_corrected(p_amp, jnp.diag(jnp.exp(2*e)), T_hat)
grad_xy = jax.grad(psi_xy)
hess_xy = jax.hessian(psi_xy)

LAM_MAX = 1.4
r = jnp.linspace(0, jnp.log(LAM_MAX)/jnp.sqrt(2/3), 41); th = jnp.linspace(0, 2*jnp.pi, 73)
R, TH = jnp.meshgrid(r, th, indexing="ij"); X, Y = R*jnp.cos(TH), R*jnp.sin(TH)
P = jnp.stack([X.ravel(), Y.ravel()], 1)
Psi = jax.vmap(psi_xy, (0, None))(P, T_HAT).reshape(X.shape)
G = jax.vmap(grad_xy, (0, None))(P, T_HAT).reshape(*X.shape, 2)
H = jax.vmap(hess_xy, (0, None))(P, T_HAT).reshape(*X.shape, 2, 2)

H0 = hess_xy(jnp.zeros(2), T_HAT)
print("grad at origin:", grad_xy(jnp.zeros(2), T_HAT))
print("Hessian at origin:\n", H0)
print("max |H_xy - H_yx|:", float(jnp.max(jnp.abs(H[..., 0, 1] - H[..., 1, 0]))))

# probe point P0: between tension and pure shear
P0 = jnp.array([0.17*np.cos(np.pi/9), 0.17*np.sin(np.pi/9)])
g0, h0 = grad_xy(P0, T_HAT), hess_xy(P0, T_HAT)
print("P0", P0, "grad", g0, "H", h0)

i_ten, i_com = 0, 12   # theta = 0 deg, 60 deg
fig = plt.figure(figsize=(16, 10.5))
panels = [
    (Psi,            r"$\Psi$  (the bowl)",                         "viridis"),
    (G[..., 0],      r"$\partial\Psi/\partial x$  (slope in $x$)",  "coolwarm"),
    (G[..., 1],      r"$\partial\Psi/\partial y$  (slope in $y$)",  "coolwarm"),
    (H[..., 0, 0],   r"$H_{xx}$: slope of $\partial\Psi/\partial x$ in $x$", "magma"),
    (H[..., 0, 1],   r"$H_{xy} = H_{yx}$: cross slope",              "coolwarm"),
    (H[..., 1, 1],   r"$H_{yy}$: slope of $\partial\Psi/\partial y$ in $y$", "magma"),
]
probe_val = [psi_xy(P0, T_HAT), g0[0], g0[1], h0[0, 0], h0[0, 1], h0[1, 1]]
d = 0.11
for n, (Z, ttl, cm) in enumerate(panels):
    ax = fig.add_subplot(2, 3, n+1, projection="3d"); ax.computed_zorder = False
    Zn = np.asarray(Z)
    if cm == "coolwarm":
        m = np.abs(Zn).max(); kw = dict(vmin=-m, vmax=m)
    else:
        kw = {}
    ax.plot_surface(X, Y, Z, cmap=cm, rstride=1, cstride=1, linewidth=0.12, edgecolor="k", alpha=0.72, zorder=1, **kw)
    ax.plot(X[:, i_ten], Y[:, i_ten], Z[:, i_ten], color="tab:red", lw=2.5, zorder=10, label="uniaxial tension")
    ax.plot(X[:, i_com], Y[:, i_com], Z[:, i_com], color="tab:blue", lw=2.5, zorder=10, label="uniaxial compression")
    ax.scatter([float(P0[0])], [float(P0[1])], [float(probe_val[n])], color="k", s=28, zorder=12)
    # tangent segments that show the Hessian as slopes of the slope surfaces
    if n == 1:   # slope of dPsi/dx along y  = H_xy
        ax.plot([P0[0], P0[0]], [P0[1]-d, P0[1]+d], [g0[0]-d*h0[0,1], g0[0]+d*h0[0,1]], color="k", lw=3, zorder=13)
    if n == 2:   # slope of dPsi/dy along x  = H_yx
        ax.plot([P0[0]-d, P0[0]+d], [P0[1], P0[1]], [g0[1]-d*h0[1,0], g0[1]+d*h0[1,0]], color="k", lw=3, zorder=13)
    if n == 0:   # gradient arrows: tangent lines of the bowl in x and y
        ax.plot([P0[0]-d, P0[0]+d], [P0[1], P0[1]], [probe_val[0]-d*g0[0], probe_val[0]+d*g0[0]], color="k", lw=3, zorder=13)
        ax.plot([P0[0], P0[0]], [P0[1]-d, P0[1]+d], [probe_val[0]-d*g0[1], probe_val[0]+d*g0[1]], color="k", lw=3, zorder=13)
        ax.legend(loc="upper left", fontsize=8)
    ax.set_xlabel(r"$x = r\cos\theta$"); ax.set_ylabel(r"$y = r\sin\theta$")
    ax.set_title(ttl, fontsize=11.5); ax.view_init(elev=24, azim=-125)
    ax.tick_params(labelsize=7)

fig.subplots_adjust(left=0.01, right=0.99, top=0.9, bottom=0.02, wspace=0.02, hspace=0.08)
fig.savefig("hessian_view_branch3.png", dpi=150)
print("---------------------------------------------------------------------------")
print("Image saved as: hessian_view_branch3.png")

## Test 5: The branch tangent vs. the boxed formula

In [ ]:
# Setup

key_test5 = jax.random.PRNGKey(42)
k1_test5, k2_test5 = jax.random.split(key_test5, 2) 

branch_k = 2
psi_params_test5 = psi_params_amplified[branch_k]

T_hat_test5 = 0.5

In [ ]:
# Random F
perturbation_test5 = jax.random.normal(k1_test5, (3, 3)) * 0.1
F_test5 = jnp.eye(3) + perturbation_test5
if jnp.linalg.det(F_test5) <= 0:
    print("Warning: Random F_test5 has non-positive determinant.")
else: 
    print("Random F_test5 is valid with positive determinant.")
print("F_test5:\n", F_test5)
print("Determinant of F_test5:", jnp.linalg.det(F_test5))

In [ ]:
# Isochoric C_bar and J
C_bar_test5, J_test5 = vc.isochoric_C(vc.right_cauchy_green(F_test5))
print("C_bar_test5:\n", C_bar_test5)
print("Sanity check (det of C_bar_test5):", jnp.abs(jnp.linalg.det(C_bar_test5)-1))
print("How close to symmetric is C_bar_test5?:", jnp.max(jnp.abs(C_bar_test5 - C_bar_test5.T)))

In [ ]:
# Path A: AD tangent
C_AD_test5 = 4*jax.hessian(lambda C: vc.psi_corrected(psi_params_test5, C, T_hat_test5))(C_bar_test5)
print("First component of C_AD_test5 (Hessian):\n", C_AD_test5[0, 0, 0, 0])
print("Shape of C_AD_test5:", C_AD_test5.shape)

In [ ]:
# Path B: hand derived tangent

#B1: Compute the invariants of C_bar_test5
I1_bar_test5, I2_bar_test5 = vc.invariants(C_bar_test5)
print("I1_bar_test5:", I1_bar_test5)
print("I2_bar_test5:", I2_bar_test5)
C_bar_test5_minus1=jnp.linalg.inv(C_bar_test5)
C_bar_test5_minus2=C_bar_test5_minus1 @ C_bar_test5_minus1

#B2: Compute the derivatives of psi with respect to the invariants 
dpsi_dI1_test5, dpsi_dI2_test5 = jax.grad(vc.psi_raw, argnums=(1, 2))(psi_params_test5, I1_bar_test5, I2_bar_test5, T_hat_test5)
(d2psi_dI1I1_test5, d2psi_dI1I2_test5), (d2psi_dI2I1_test5, d2psi_dI2I2_test5) = jax.hessian(vc.psi_raw, argnums=(1, 2))(psi_params_test5, I1_bar_test5, I2_bar_test5, T_hat_test5)
print("Schwarz test:", float(d2psi_dI1I2_test5 - d2psi_dI2I1_test5))

#B3: 